
# Temporal Trend Analysis (Beehive Acoustic Features)

## Aim
This notebook explores **temporal patterns** in extracted acoustic features (RMS, spectral features, ZCR, MFCC summaries) and their relationship with environmental variables (temperature and humidity).

The goal is to identify:
- daily (diurnal) patterns
- long-term trends
- potential behavioural cycles within the hive



## Notes
All features were previously extracted and aligned with timestamps.

Because signals were normalised during preprocessing, analysis focuses on **relative changes over time** rather than absolute amplitudes.


In [ ]:

import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Configuration

In [ ]:

FEATURE_FOLDER = "data/features"
OUTPUT_FOLDER = "data/analysis"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


## Load and merge datasets

In [ ]:

from functools import reduce

files = glob(os.path.join(FEATURE_FOLDER, "*.csv"))

dfs = [pd.read_csv(f, parse_dates=["timestamp"]) for f in files]

df = reduce(lambda left, right: pd.merge(left, right, on="timestamp", how="inner"), dfs)

df = df.dropna()

df.head()


## Create time-based features

In [ ]:

df["hour"] = df["timestamp"].dt.hour
df["date"] = df["timestamp"].dt.date


## Average feature by hour (diurnal pattern)

In [ ]:

hourly = df.groupby("hour").mean(numeric_only=True)

hourly.head()


## Plot hourly trends

In [ ]:

plt.figure(figsize=(12,6))

for col in hourly.columns:
    if col not in ["Temperatura (°C)", "Wilgotność (%)"]:
        plt.plot(hourly.index, hourly[col], label=col)

plt.xlabel("Hour of Day")
plt.ylabel("Mean Value")
plt.title("Average Acoustic Features by Hour")
plt.legend()
plt.show()


## Daily trend

In [ ]:

daily = df.groupby("date").mean(numeric_only=True)

plt.figure(figsize=(12,5))

for col in daily.columns:
    if col not in ["Temperatura (°C)", "Wilgotność (%)"]:
        plt.plot(daily.index, daily[col], label=col)

plt.xticks(rotation=45)
plt.title("Daily Feature Trends")
plt.legend()
plt.show()


## Feature vs Environment over time

In [ ]:

sample = df.iloc[:10000]

plt.figure(figsize=(14,5))
plt.plot(sample["timestamp"], sample["rms"], label="RMS")
plt.plot(sample["timestamp"], sample["Temperatura (°C)"], label="Temp")
plt.plot(sample["timestamp"], sample["Wilgotność (%)"], label="Humidity")

plt.legend()
plt.title("Feature vs Environment (Sample)")
plt.show()



## Interpretation

Temporal analysis allows identification of:

- **diurnal cycles** → differences between day and night activity
- **environmental influence** → whether temperature/humidity correlate with acoustic behaviour
- **stable vs dynamic periods** → identifying calm vs active intervals

These insights help understand behavioural patterns within the hive and support further modelling.
